## Init config

In [1]:
import torch
from common_functions_python import set_config_file, test_function

import warnings

# Suppress the specific UserWarning
warnings.filterwarnings("ignore", category=UserWarning, message=".*copy constructor.*")
warnings.filterwarnings("ignore", category=UserWarning)

config_file = {
                'name': 'Pretrained',
                'datasets': ['ABC'],
                'bidirectional_lstm': False,
                'lstm_dropout': 0.2,
                'mlp_dropout': 0.2,
                'lr': 0.00005,
                'step_size': 5,
                'gamma': 0.5,
                'weight_decay': 0.01,
                'hidden_dim': 512,
                'num_layers': 3,
                'batch_size': 8, 
                'frame_frequency': 2,
                'num_epoch': 20,
                'num_workers': 4,
                'concatenate': True
                }

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print('device: ', device)
set_config_file(config_file, device)
# test_function()

/media/osero/SamsungSSD/miniconda_files/conda/envs/dinov2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device:  cuda


In [2]:
import contextlib
import gc
from common_functions_dino_python import train_loop_dino
from common_functions_heatmap_python import train_loop_heatmap
from common_functions_only_heatmap_python import train_loop_only_heatmap

@contextlib.contextmanager
def clear_memory():
    try:
        yield
    finally:
        gc.collect()

datasets_list = [
    # ['deephand_left'],
    # ['deephand_right'],
    # ['dino_left_large'],
    # ['dino_left_small'],
    # ['dino_right_large']
    # ['dino_right_small'],
    # ['dino_face_small'],
    # ['deephand_left', 'dino_left_large'],
    # ['deephand_left', 'dino_left_large', 'dino_right_large', 'dino_face_small'],
    # ['deephand_left', 'dino_left_large', 'dino_face_small', 'dino_right_small', 'heatmap_3d'],
    # ['deephand_left', 'dino_left_large', 'dino_face_small', 'dino_right_small', 'heatmap'],
    
    # ['heatmap'],
    # ['heatmap_3d'],
    # ['heatmap_limb'],
    # ['dino_left_large', 'dino_right_small', 'dino_face_small'],
    # ['deephand_left', 'deephand_right', 'dino_left_large', 'dino_right_small', 'dino_face_small'],
    # ['heatmap_3d_limb'],
    # ['dino_left_large', 'dino_right_small', 'dino_face_small'],
    # ['dino_left_large', 'dino_right_large', 'dino_face_small'],
    # ['deephand_left', 'dino_left_large', 'dino_right_large', 'dino_face_small', 'heatmap_3d'],
    # ['dino_left_small_trained'],
    # ['dino_left_large'],
    # ['dino_left_large_trained_mixed'],
    # ['dino_right_large'],
    # ['dino_right_large_trained_mixed'],
    ['dino_face_large_trained_mixed'],
]

# dropout_list = [0.05, 0.1, 0.15, 0.2]
dropout_list = [0.1]

frame_frequency_list = [2]

bidirectional_lstm_list = [False]

batch_size_list = [64]

# hidden_dim_list = [2048, 1024, 512]

hidden_dim_list = [1024]

num_layers_list = [2]
#num_layers_list = [3]

# weight_decay_list = [0.1, 0.05, 0.01, 0.005]
weight_decay_list = [0.01]

concatenate_list = [True]

for datasets in datasets_list:
    for dropout in dropout_list:
        for frame_frequency in frame_frequency_list:
            for bidirectional_lstm in bidirectional_lstm_list:
                for batch_size in batch_size_list:
                    for hidden_dim in hidden_dim_list:
                        for num_layers in num_layers_list:
                            for weight_decay in weight_decay_list:
                                for concatenate in concatenate_list:
                                    gc.collect()
                                    with clear_memory():   
                                        config_file['datasets'] = datasets
                                        config_file['lstm_dropout'] = dropout
                                        config_file['mlp_dropout'] = dropout
                                        config_file['frame_frequency'] = frame_frequency
                                        config_file['bidirectional_lstm'] = bidirectional_lstm
                                        config_file['batch_size'] = batch_size
                                        config_file['hidden_dim'] = hidden_dim
                                        config_file['num_layers'] = num_layers
                                        config_file['weight_decay'] = weight_decay
                                        config_file['concatenate'] = concatenate
                                        set_config_file(config_file, device)
                                        print(config_file)
                                        if any('heatmap' in s for s in datasets) and len(datasets) == 1:
                                            if concatenate == concatenate_list[0] and bidirectional_lstm == bidirectional_lstm_list[0]:
                                                train_loop_only_heatmap()
                                        elif any('heatmap' in s for s in datasets) and len(datasets) != 1:
                                            train_loop_heatmap()
                                        else:
                                            train_loop_dino()


{'name': 'Pretrained', 'datasets': ['dino_face_large_trained_mixed'], 'bidirectional_lstm': False, 'lstm_dropout': 0.1, 'mlp_dropout': 0.1, 'lr': 5e-05, 'step_size': 5, 'gamma': 0.5, 'weight_decay': 0.01, 'hidden_dim': 1024, 'num_layers': 2, 'batch_size': 64, 'frame_frequency': 2, 'num_epoch': 20, 'num_workers': 4, 'concatenate': True}
datasets:  ['dino_face_large_trained_mixed']
input_dim:  1024  num_classes:  744
train_dataset size:  18018
test_dataset size:  4524
train_loader pickle_file_name_list:  ['/media/osero/SamsungSSD/pickles/features_face_frames_large_trained_mixed_train.pickle']
test_loader pickle_file_name_list:  ['/media/osero/SamsungSSD/pickles/features_face_frames_large_trained_mixed_test.pickle']


Epoch [1/20]: 100%|██████████| 282/282 [00:12<00:00, 23.06it/s, acc=0, loss=6.57]     


Time: 2025-01-09_21-13-15 Epoch [1], Avg loss: 6.6018, Avg accuracy: 0.9142
Accuracy of the network on the 4524 test video: 1.2821 %, top5: 3.3820 %, avg_loss: 0.10336758339963689, total: 4524


Epoch [2/20]: 100%|██████████| 282/282 [00:12<00:00, 23.29it/s, acc=0.0588, loss=5.17]

Time: 2025-01-09_21-13-29 Epoch [2], Avg loss: 5.8981, Avg accuracy: 3.7830


...

Time: 2025-01-09_21-17-39 Epoch [19], Avg loss: 0.8314, Avg accuracy: 97.2739


Accuracy of the network on the 4524 test video: 21.1760 %, top5: 39.1689 %, avg_loss: 0.0682956454081413, total: 4524


Epoch [20/20]: 100%|██████████| 282/282 [00:12<00:00, 22.51it/s, acc=1, loss=0.862]    

Time: 2025-01-09_21-17-54 Epoch [20], Avg loss: 0.7971, Avg accuracy: 97.6341


Accuracy of the network on the 4524 test video: 21.3307 %, top5: 39.6331 %, avg_loss: 0.06805592369548416, total: 4524


## Heatmap Test

In [ ]:
import pickle
import moviepy as mpy
import copy as cp
from pyskl_lib import *
import torch

pickle_file_name = '/media/osero/SamsungSSD/pickles/bsign22_heatmap_format_full_test.pkl'
pickle_file = open(pickle_file_name, 'rb')
annotations = pickle.load(pickle_file)
annotation = annotations[20]
annotation['keypoint'] = annotation['keypoint'][:,:, :15, :]
annotation['keypoint_score'] = annotation['keypoint_score'][:,:, :15]

my_keypoint_heatmap = get_pseudo_heatmap(cp.deepcopy(annotation))
my_keypoint_mapvis = vis_heatmaps(my_keypoint_heatmap)
my_keypoint_mapvis = [add_label(f, annotation['frame_dir'].split('/')[-2] + '/' + annotation['frame_dir'].split('/')[-1]) for f in my_keypoint_mapvis]
my_vid = mpy.ImageSequenceClip(my_keypoint_mapvis, fps=24)
my_vid.display_in_notebook()

MoviePy - Building video __temp__.mp4.
MoviePy - Writing video __temp__.mp4



MoviePy - Done !
MoviePy - video ready __temp__.mp4


## Analyze Results

In [4]:
# from sklearn.metrics import classification_report, confusion_matrix
# import pandas as pd

# my_anno = torch.load("/home/osero/Desktop/CMPE/dinov2/classsification/lstm/lstm_results/DINO_features_sum_2024-12-24_01-11-48.pth")
# avc = 2

# print(classification_report(my_anno['test_prediction_results'][1], my_anno['test_prediction_results'][0]))


# df = pd.DataFrame(data)

# # Get unique labels
# unique_labels = df['true_labels'].unique()

# # Calculate and print accuracy for each label
# print("Accuracy for each label:")
# for label in unique_labels:
#     # Filter rows where the true label is the current label
#     label_mask = df['true_labels'] == label
    
#     # Calculate accuracy for the current label
#     label_accuracy = accuracy_score(
#         df.loc[label_mask, 'true_labels'], 
#         df.loc[label_mask, 'predicted_labels']
#     )
    
#     print(f"Label '{label}': {label_accuracy:.2f}")

In [5]:
# import pandas as pd
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# cm = confusion_matrix(my_anno['test_prediction_results'][1], my_anno['test_prediction_results'][0])
# df_cm = pd.DataFrame(
#     cm
# )

# cm = confusion_matrix(my_anno['test_prediction_results'][0], my_anno['test_prediction_results'][1])

# # Step 2: Display the confusion matrix
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1, 2, 3])
# disp.plot(cmap="viridis")  # You can use other colormaps like 'plasma' or 'Blues'

# # Optional: Customize the plot
# import matplotlib.pyplot as plt
# plt.title("Confusion Matrix")
# plt.xlabel("Predicted Labels")
# plt.ylabel("True Labels")
# plt.show()

In [6]:
# from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

# y_true = my_anno['test_prediction_results'][1]
# y_pred = my_anno['test_prediction_results'][0]
# # Compute confusion matrix
# cm = confusion_matrix(y_true, y_pred)
# print("Confusion Matrix:")
# print(cm)

# # Extract confusion matrix elements
# tn, fp, fn, tp = cm.ravel()
# print("\nConfusion Matrix Elements:")
# print(f"True Negatives (TN): {tn}")
# print(f"False Positives (FP): {fp}")
# print(f"False Negatives (FN): {fn}")
# print(f"True Positives (TP): {tp}")

# # Calculate metrics
# accuracy = accuracy_score(y_true, y_pred)
# precision = precision_score(y_true, y_pred)
# recall = recall_score(y_true, y_pred)
# f1 = f1_score(y_true, y_pred)

# print("\nMetrics:")
# print(f"Accuracy: {accuracy:.2f}")
# print(f"Precision: {precision:.2f}")
# print(f"Recall: {recall:.2f}")
# print(f"F1 Score: {f1:.2f}")

# # Alternatively, use classification report
# print("\nClassification Report:")
# print(classification_report(y_true, y_pred))